# PointsX — Synthetic Dataset Generation (SMPL-X + Blender)

**Goal:** Generate ~10 000 photorealistic synthetic images with 25 custom body-measurement keypoints and ground-truth measurement JSON files.

**Pipeline:**
1. Install dependencies (smplx, smpl-anthropometry, Blender)
2. Download SMPL-X model files (requires free registration)
3. Download assets: Poly Haven HDRIs, SMPLitex skin textures
4. Run SMPL-X phase: generate 500 bodies × 3 poses → OBJ + landmarks JSON
5. Run Blender phase: render front + side views → 10 k JPEGs + YOLO labels
6. Package dataset for YOLO training

**Runtime:** ~6–8 hours on a Colab T4 GPU (Blender Cycles, 256 samples/image).

> **Before starting:** Mount your Google Drive and point `DRIVE_ROOT` to your project folder.

## 0. Configuration

In [ ]:
# ── User settings ──────────────────────────────────────────────────────────
DRIVE_ROOT   = "/content/drive/MyDrive/PointsX"   # Change to your Drive path
N_BODIES     = 500    # unique body shapes (each × 3 poses = 1500 total)
SEED         = 42
BLENDER_JOBS = 1      # parallel Blender instances (1 on Colab)
USE_GPU      = True   # set False if Colab GPU not available

# Local (fast SSD) working dirs inside Colab VM
WORK_DIR     = "/content/pointsx_work"
ASSETS_DIR   = f"{WORK_DIR}/assets"
MODEL_DIR    = f"{WORK_DIR}/models/smplx"
OUT_DIR      = f"{WORK_DIR}/data/synthetic-pose"

print("Config OK")

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs(WORK_DIR,   exist_ok=True)
os.makedirs(ASSETS_DIR, exist_ok=True)
os.makedirs(MODEL_DIR,  exist_ok=True)
os.makedirs(OUT_DIR,    exist_ok=True)
print("Directories created.")

## 2. Install Python dependencies

In [ ]:
%%bash
pip install -q \
    smplx==0.1.28 \
    numpy \
    torch torchvision \
    tqdm

# Install SMPL-Anthropometry from GitHub (pip package name may vary)
pip install -q git+https://github.com/DavidBoja/SMPL-Anthropometry.git || \
    pip install -q smpl-anthropometry

echo "Python deps installed."

## 3. Install Blender

In [ ]:
%%bash
# Install Blender 3.6 LTS (stable with Python 3.10)
BLENDER_VER="3.6.9"
BLENDER_URL="https://download.blender.org/release/Blender3.6/blender-${BLENDER_VER}-linux-x64.tar.xz"

if [ ! -f /content/blender/blender ]; then
    echo "Downloading Blender ${BLENDER_VER}..."
    wget -q "$BLENDER_URL" -O /tmp/blender.tar.xz
    tar -xf /tmp/blender.tar.xz -C /content/
    mv /content/blender-${BLENDER_VER}-linux-x64 /content/blender
    rm /tmp/blender.tar.xz
    echo "Blender installed."
else
    echo "Blender already present."
fi

/content/blender/blender --version | head -1

In [ ]:
BLENDER_EXE = "/content/blender/blender"

# Install smplx inside Blender's embedded Python so blender_render.py can import it
import subprocess
blender_pip = f"{BLENDER_EXE}/../3.6/python/bin/python3.10"
subprocess.run(
    [blender_pip, "-m", "pip", "install", "-q", "smplx", "numpy"],
    check=False
)
print("smplx installed in Blender Python.")

## 4. Install PointsX package

In [ ]:
%%bash
# Clone or pull from Drive
REPO_SRC="/content/drive/MyDrive/PointsX"

if [ -d "$REPO_SRC" ]; then
    cp -r "$REPO_SRC/src" /content/pointsx_src
    cd /content/pointsx_src && pip install -q -e .
    echo "PointsX installed from Drive copy."
else
    echo "WARNING: $REPO_SRC not found on Drive."
    echo "Please upload your PointsX/src folder to Google Drive first."
fi

## 5. Download SMPL-X model files

You must **manually** download `SMPLX_MALE.npz` and `SMPLX_FEMALE.npz` from:
👉 https://smpl-x.is.tue.mpg.de/ (free registration required)

Upload the files to your Drive folder and run the cell below to copy them.

In [ ]:
import shutil, os
from pathlib import Path

smplx_drive = Path(DRIVE_ROOT) / "models" / "smplx"
smplx_local = Path(MODEL_DIR)

for fname in ["SMPLX_MALE.npz", "SMPLX_FEMALE.npz", "SMPLX_NEUTRAL.npz"]:
    src = smplx_drive / fname
    dst = smplx_local / fname
    if src.exists() and not dst.exists():
        shutil.copy2(src, dst)
        print(f"Copied {fname}")
    elif dst.exists():
        print(f"Already present: {fname}")
    else:
        print(f"MISSING: {fname}  ← upload to {smplx_drive}")

## 6. Download Assets

### 6a. Poly Haven HDRIs (CC0 free)

In [ ]:
import os, subprocess
from pathlib import Path

hdri_dir = Path(ASSETS_DIR) / "hdri"
hdri_dir.mkdir(parents=True, exist_ok=True)

# 20 indoor HDRIs from Poly Haven (1K resolution for speed)
POLY_HAVEN_HDRIS = [
    "abandoned_slipway",
    "artist_workshop",
    "bathroom",
    "blue_photo_studio",
    "carpentry_shop_01",
    "chinese_garden",
    "empty_warehouse_01",
    "gym",
    "hotel_room",
    "indoor_cycling",
    "living_room",
    "modern_bathroom",
    "office",
    "old_hall",
    "photo_studio_01",
    "restaurant",
    "studio_small_03",
    "surgery_room",
    "table_mountain_1",
    "urban_street_01",
]

downloaded = 0
for hdri_name in POLY_HAVEN_HDRIS:
    out_path = hdri_dir / f"{hdri_name}_1k.hdr"
    if out_path.exists():
        continue
    url = f"https://dl.polyhaven.org/file/ph-assets/HDRIs/hdr/1k/{hdri_name}_1k.hdr"
    result = subprocess.run(
        ["wget", "-q", "-O", str(out_path), url],
        capture_output=True
    )
    if result.returncode == 0:
        downloaded += 1
    else:
        print(f"  Failed: {hdri_name}")

total = len(list(hdri_dir.glob("*.hdr")))
print(f"HDRIs ready: {total} files (downloaded {downloaded} new)")

### 6b. Skin Textures

Option A: Use SMPLitex (requires running the diffusion model — slow)  
Option B: Use a pre-built 30-texture library from Drive (recommended)

The cell below generates simple procedural fallback textures if neither is available.

In [ ]:
import numpy as np
from pathlib import Path
from PIL import Image

tex_dir = Path(ASSETS_DIR) / "textures"
tex_dir.mkdir(parents=True, exist_ok=True)

# Check if Drive has pre-built textures
drive_tex = Path(DRIVE_ROOT) / "assets" / "textures"
if drive_tex.exists():
    import shutil
    for f in drive_tex.glob("skin_*.png"):
        dst = tex_dir / f.name
        if not dst.exists():
            shutil.copy2(f, dst)
    print(f"Copied {len(list(tex_dir.glob('skin_*.png')))} textures from Drive.")
else:
    # Generate 30 procedural skin-tone textures as fallback
    # Fitzpatrick scale: 6 tones × 5 variants = 30
    SKIN_TONES = [
        (255, 224, 196),  # Type I   — very fair
        (241, 194, 125),  # Type II  — fair
        (224, 172, 105),  # Type III — medium
        (198, 134,  66),  # Type IV  — olive
        (141,  85,  36),  # Type V   — brown
        ( 80,  50,  20),  # Type VI  — dark
    ]
    tex_size = 1024
    rng = np.random.default_rng(42)
    idx = 1
    for base_rgb in SKIN_TONES:
        for variant in range(5):
            # Add per-pixel noise for variation
            img_arr = np.ones((tex_size, tex_size, 3), dtype=np.uint8)
            for c, val in enumerate(base_rgb):
                noise = rng.integers(-15, 15, (tex_size, tex_size))
                img_arr[:, :, c] = np.clip(val + noise, 0, 255)
            img = Image.fromarray(img_arr, mode="RGB")
            img.save(str(tex_dir / f"skin_{idx:02d}.png"))
            idx += 1
    print(f"Generated {idx-1} procedural skin textures (fallback mode).")
    print("For photorealistic results, add SMPLitex UV maps to Drive/assets/textures/")

### 6c. Clothing Meshes

Clothing OBJ files must be built manually in Blender (Shrinkwrap modifier on A-pose SMPL-X mesh).  
If not present, Blender will render bare bodies (useful for initial testing).

In [ ]:
from pathlib import Path
cloth_dir = Path(ASSETS_DIR) / "clothing"
cloth_dir.mkdir(parents=True, exist_ok=True)

# Copy from Drive if available
drive_cloth = Path(DRIVE_ROOT) / "assets" / "clothing"
if drive_cloth.exists():
    import shutil
    for f in drive_cloth.glob("*.obj"):
        dst = cloth_dir / f.name
        if not dst.exists():
            shutil.copy2(f, dst)
    print(f"Copied {len(list(cloth_dir.glob('*.obj')))} clothing meshes from Drive.")
else:
    print("No clothing meshes found — rendering bare bodies.")
    print(f"Add OBJ files to {drive_cloth} for clothed renders.")

## 7. SMPL-X Phase — Generate Bodies

In [ ]:
import sys
sys.path.insert(0, "/content/pointsx_src/src")

from pointsx.synthetic.body_generator import generate_body_samples

samples = generate_body_samples(n_bodies=N_BODIES, seed=SEED)
print(f"Generated {len(samples)} body+pose variants")
print(f"  Male: {sum(1 for s in samples if s.sex == 'male')}")
print(f"  Female: {sum(1 for s in samples if s.sex == 'female')}")

# Quick stats
import numpy as np
heights = [s.target_height_cm for s in samples]
print(f"  Height range: {min(heights):.0f}–{max(heights):.0f} cm  mean={np.mean(heights):.0f}")

In [ ]:
from pathlib import Path
from pointsx.synthetic.pipeline import run_smplx_phase

manifest = run_smplx_phase(
    samples,
    model_dir=Path(MODEL_DIR),
    out_dir=Path(OUT_DIR),
)
print(f"SMPL-X phase done: {len(manifest)} entries in manifest")

## 8. Validate SMPL-X Output

In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path

meas_dir = Path(OUT_DIR) / "measurements"
all_meas = []
for p in sorted(meas_dir.glob("*.json"))[:N_BODIES]:  # one per body (first pose)
    all_meas.append(json.loads(p.read_text()))

if all_meas:
    fields = ["actual_height_cm", "waist_circumference_cm", "hips_circumference_cm",
              "chest_circumference_cm", "thigh_circumference_cm"]
    fig, axes = plt.subplots(1, len(fields), figsize=(18, 3))
    for ax, field in zip(axes, fields):
        vals = [m.get(field, 0) for m in all_meas]
        ax.hist(vals, bins=30, color="steelblue", edgecolor="white")
        ax.set_title(field.replace("_cm", "").replace("_", " "))
        ax.set_xlabel("cm")
    plt.tight_layout()
    plt.show()
    print(f"Validated {len(all_meas)} measurement files.")
else:
    print("No measurements found — check SMPL-X phase.")

## 9. Blender Phase — Render All Views

In [ ]:
# Test single render first to verify Blender setup
import subprocess, json
from pathlib import Path

# Write a mini manifest with just 1 body
test_manifest = [manifest[0]] if manifest else []
test_manifest_path = Path(OUT_DIR) / "test_manifest.json"
test_manifest_path.write_text(json.dumps(test_manifest, indent=2))

blender_script = "/content/pointsx_src/src/pointsx/synthetic/blender_render.py"

cmd = [
    BLENDER_EXE,
    "--background",
    "--python", blender_script,
    "--",
    "--manifest",  str(test_manifest_path),
    "--out-dir",   OUT_DIR,
    "--assets",    ASSETS_DIR,
    "--start-idx", "0",
    "--end-idx",   "1",
]
if USE_GPU:
    cmd.append("--gpu")

print("Running test render (1 body × 2 views)...")
result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
print(result.stdout[-3000:])  # last 3k chars
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])

In [ ]:
# Visualise the test render
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

for split in ("train", "val"):
    imgs = sorted(Path(OUT_DIR, split, "images").glob("s00001_*.jpg"))
    for img_path in imgs:
        img = Image.open(img_path)
        plt.figure(figsize=(4, 4))
        plt.imshow(img)
        plt.title(img_path.name)
        plt.axis("off")
        plt.show()

In [ ]:
# Full render run
import json, subprocess
from pathlib import Path

# Write full manifest
manifest_path = Path(OUT_DIR) / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
print(f"Manifest: {len(manifest)} entries → {manifest_path}")

blender_script = "/content/pointsx_src/src/pointsx/synthetic/blender_render.py"

cmd = [
    BLENDER_EXE,
    "--background",
    "--python", blender_script,
    "--",
    "--manifest",  str(manifest_path),
    "--out-dir",   OUT_DIR,
    "--assets",    ASSETS_DIR,
]
if USE_GPU:
    cmd.append("--gpu")

print("Starting full render... (this will take several hours)")
result = subprocess.run(cmd, capture_output=True, text=True, timeout=28800)  # 8h max
print(result.stdout[-5000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
print(f"\nBlender exited with code {result.returncode}")

## 10. Write dataset.yaml

In [ ]:
from pathlib import Path
from pointsx.synthetic.pipeline import write_dataset_yaml

yaml_path = write_dataset_yaml(Path(OUT_DIR))
print(yaml_path.read_text())

## 11. Dataset Statistics

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

out = Path(OUT_DIR)
print("=== Dataset Summary ===")
for split in ("train", "val"):
    imgs   = list((out / split / "images").glob("*.jpg"))
    labels = list((out / split / "labels").glob("*.txt"))
    print(f"  {split}: {len(imgs):>5} images | {len(labels):>5} labels")

# Label statistics
all_labels = list((out / "train" / "labels").glob("*.txt")) + \
             list((out / "val"   / "labels").glob("*.txt"))

vis_counts = []
for lp in all_labels:
    line = lp.read_text().strip()
    if not line:
        continue
    parts = line.split()
    # Each keypoint: x y v  → v is at index 5 + i*3 + 2
    vs = [int(parts[5 + i*3 + 2]) for i in range(25) if 5 + i*3 + 2 < len(parts)]
    vis_counts.append(sum(1 for v in vs if v >= 1))

if vis_counts:
    vis_arr = np.array(vis_counts)
    print(f"\nVisible keypoints per image:")
    print(f"  Mean: {vis_arr.mean():.1f}  Min: {vis_arr.min()}  Max: {vis_arr.max()}")
    print(f"  Images with >= 20 visible: {(vis_arr >= 20).sum()} ({(vis_arr>=20).mean()*100:.1f}%)")

    plt.figure(figsize=(8, 3))
    plt.hist(vis_arr, bins=range(0, 27), color="steelblue", edgecolor="white")
    plt.xlabel("Visible keypoints per image")
    plt.ylabel("Count")
    plt.title("Keypoint Visibility Distribution")
    plt.tight_layout()
    plt.show()

In [ ]:
# Visualise a few random samples with landmark overlay
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

LANDMARK_NAMES_SHORT = [
    "head","chin","neck_b","L_sh","R_sh","chest",
    "L_ax","R_ax","mid_bk","navel",
    "L_el","R_el","L_wa","R_wa","low_bk",
    "L_hip","R_hip","crotch","glute",
    "L_wr","R_wr","L_kn","R_kn","L_an","R_an",
]
COLORS = {2: "#00ff88", 1: "#ffaa00", 0: "#ff3333"}

img_dir   = Path(OUT_DIR) / "train" / "images"
label_dir = Path(OUT_DIR) / "train" / "labels"

sample_imgs = random.sample(sorted(img_dir.glob("*.jpg")), min(6, len(list(img_dir.glob("*.jpg")))))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_path in zip(axes.flat, sample_imgs):
    label_path = label_dir / img_path.with_suffix(".txt").name
    img = np.array(Image.open(img_path))
    ax.imshow(img)
    ax.set_title(img_path.stem, fontsize=8)
    ax.axis("off")

    if not label_path.exists():
        continue
    parts = label_path.read_text().strip().split()
    if len(parts) < 5 + 25*3:
        continue

    H, W = img.shape[:2]
    # Draw bbox
    cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
    x0 = (cx - bw/2) * W
    y0 = (cy - bh/2) * H
    rect = patches.Rectangle((x0, y0), bw*W, bh*H,
                               linewidth=1, edgecolor="white", facecolor="none")
    ax.add_patch(rect)

    # Draw keypoints
    for i in range(25):
        offset = 5 + i * 3
        kx = float(parts[offset])   * W
        ky = float(parts[offset+1]) * H
        v  = int(parts[offset+2])
        if v == 0:
            continue
        ax.plot(kx, ky, "o", color=COLORS[v], markersize=4)
        ax.text(kx+2, ky-2, LANDMARK_NAMES_SHORT[i], fontsize=4,
                color="white", bbox=dict(facecolor="black", alpha=0.4, pad=0.2))

plt.suptitle("Synthetic dataset samples with 25-keypoint annotations", fontsize=12)
plt.tight_layout()
plt.show()

## 12. Save Dataset to Google Drive

In [ ]:
%%bash
echo "Syncing dataset to Google Drive..."

DEST="/content/drive/MyDrive/PointsX/data/synthetic-pose"
mkdir -p "$DEST"

# Use rsync for efficient incremental copy
rsync -a --info=progress2 \
    /content/pointsx_work/data/synthetic-pose/ \
    "$DEST/"

echo "Done."
ls -lh "$DEST"

## 13. Quick YOLO Training Test

Train `yolo11n-pose` for 10 epochs on the synthetic data to verify label format is correct.

In [ ]:
%%bash
pip install -q ultralytics
echo "ultralytics ready"

In [ ]:
from ultralytics import YOLO
from pathlib import Path

dataset_yaml = str(Path(OUT_DIR) / "dataset.yaml")

model = YOLO("yolo11n-pose.pt")
results = model.train(
    data=dataset_yaml,
    epochs=10,
    imgsz=640,
    batch=16,
    device=0 if USE_GPU else "cpu",
    project="/content/runs",
    name="synthetic_test",
    exist_ok=True,
    verbose=False,
)
print(f"\nTest training complete.")
print(f"Best model: {results.best}")

---
## ✅ Pipeline Complete

| Output | Location |
|--------|----------|
| Train images | `data/synthetic-pose/train/images/` |
| Val images   | `data/synthetic-pose/val/images/` |
| YOLO labels  | `data/synthetic-pose/{train,val}/labels/` |
| Measurements | `data/synthetic-pose/measurements/` |
| Dataset YAML | `data/synthetic-pose/dataset.yaml` |
| Body meshes  | `data/synthetic-pose/meshes/` |

**Next steps:**
1. Fine-tune `yolo11n-pose` on synthetic + LV-MHP-v2 combined
2. Build regression training dataset from measurement JSONs
3. Train circumference regression MLP (`src/pointsx/regression/train.py`)